# When Models Lie to Please: Google Colab Runner

This notebook runs the actual, GPU-heavy 4-phase sparse interpretability research pipeline on a high-VRAM Google Colab runtime (T4, L4, or A100 GPU).

### Pipeline Steps:
1. **Git Clone/Pull**: Sets up the codebase and installs all dependencies.
2. **(Optional) Mount Google Drive**: Mounts Google Drive and symlinks the results folder so results are saved progressively. **Highly recommended to prevent data loss on disconnects!**
3. **Build Datasets**: Generates the paired logic/opinion/sycophancy datasets.
4. **Phase 1 (Feature Discovery)**: Extracts activations and finds differential SAE features.
5. **Phase 2 (Circuit Tracing)**: Builds transcoder attribution graphs and analyzes divergence.
6. **Phase 3 (Transfer Testing)**: Runs cross-condition transfer and measures representation geometry.
7. **Phase 4 (Interventions & CAST)**: Trains classifiers and tests steering/CAST mitigations.
8. **Zip & Download**: Compresses results and datasets into a ZIP file for local download and saves a copy to your Google Drive.

## Step 1: Install Dependencies and Setup

In [ ]:
# Clone repo or pull latest updates
!git clone https://github.com/ashioyajotham/when_models_lie_to_please.git || (cd when_models_lie_to_please && git pull)
%cd when_models_lie_to_please

# Install dependencies
!pip install -e ".[dev]"
!pip install git+https://github.com/google-deepmind/mishax.git

print("Setup complete!")

## Step 1b: (Optional) Mount Google Drive to Persist Results
Highly recommended! If you run this cell, all experiment results will be written directly to your Google Drive folder (`when_models_lie_to_please_results`) progressively. If your Colab session disconnects or crashes due to timeout/RAM, you will **not** lose any of your generated results or graphs.

In [ ]:
import os
import shutil

USE_GOOGLE_DRIVE = True  # Set to False to skip Drive mounting

if USE_GOOGLE_DRIVE:
    try:
        from google.colab import drive
        print("Mounting Google Drive...")
        drive.mount('/content/drive')
        
        # Create a directory in Drive for results
        drive_results_dir = "/content/drive/MyDrive/when_models_lie_to_please_results"
        os.makedirs(drive_results_dir, exist_ok=True)
        
        # Symlink experiments/results to Drive
        is_link = os.path.islink("experiments/results")
        
        if os.path.exists("experiments/results") and not is_link:
            print("Moving existing results to Google Drive...")
            for item in os.listdir("experiments/results"):
                src_item = os.path.join("experiments/results", item)
                dst_item = os.path.join(drive_results_dir, item)
                if os.path.isdir(src_item):
                    if os.path.exists(dst_item):
                        shutil.rmtree(dst_item)
                    shutil.move(src_item, dst_item)
                else:
                    if os.path.exists(dst_item):
                        os.remove(dst_item)
                    shutil.move(src_item, dst_item)
            shutil.rmtree("experiments/results")
            
        if not os.path.exists("experiments/results") and not os.path.islink("experiments/results"):
            os.symlink(drive_results_dir, "experiments/results")
            print("Google Drive mounted and experiments/results symlinked successfully!")
        else:
            print("Google Drive is already symlinked/mounted.")
    except Exception as e:
        print(f"Could not use Google Drive: {e}")
        print("Falling back to local ephemeral storage.")

## Step 2: Configure Token and Run Pipeline
Make sure to add your HuggingFace token as a secret in Colab (named `HF_TOKEN`) or enter it when prompted.

In [ ]:
import os
import glob
import json
import time

# Authenticate with HuggingFace (required for gated Gemma 3 models)
try:
    from google.colab import userdata
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
except Exception:
    os.environ["HF_TOKEN"] = input("Enter your HuggingFace Token: ").strip()

pipeline_start = time.time()

# Step 0: Build paired prompt datasets
!python experiments/scripts/build_datasets.py --output-dir data/processed --min-pairs 500

# Clear any old/corrupted activation cache to ensure fresh extraction with loaded SAEs
!rm -rf data/activations/gemma3_4b

# Step 1: Feature discovery
!python experiments/scripts/run_phase1.py --config configs/experiment_configs/phase1_features.yaml --ignore-cache

# Find latest Phase 1 run_id to pass into Phase 2/3/4
phase1_runs = sorted(glob.glob("experiments/results/phase1/*"))
if not phase1_runs:
    raise RuntimeError("Phase 1 execution did not produce any output folders.")
run_id = os.path.basename(phase1_runs[-1])
print(f"\n>>> Detected Phase 1 Run ID: {run_id} <<<\n")

# Step 2: Circuit tracing
!python experiments/scripts/run_phase2.py --config configs/experiment_configs/phase2_circuits.yaml --phase1-run {run_id}

# Step 3: Shared mechanism testing
!python experiments/scripts/run_phase3.py --config configs/experiment_configs/phase3_transfer.yaml --phase1-run {run_id}

# Step 4: Detection and mitigation
!python experiments/scripts/run_phase4.py --config configs/experiment_configs/phase4_interventions.yaml --phase1-run {run_id} --phase3-run {run_id}

# ═══════════════════════════════════════════════════════════════════════
# PIPELINE SUMMARY — always printed at the very end for visibility
# ═══════════════════════════════════════════════════════════════════════
elapsed = time.time() - pipeline_start
print("\n")
print("=" * 72)
print("  PIPELINE EXECUTION SUMMARY")
print("=" * 72)
print(f"  Total elapsed time: {elapsed/60:.1f} minutes")
print("-" * 72)

for phase_name in ["phase1", "phase2", "phase3", "phase4"]:
    base = f"experiments/results/{phase_name}"
    if not os.path.exists(base):
        print(f"\n  [MISSING] {phase_name.upper()}: No results directory found")
        continue
    runs = sorted(os.listdir(base))
    if not runs:
        print(f"\n  [MISSING] {phase_name.upper()}: Empty results directory")
        continue
    latest = runs[-1]
    summary_path = os.path.join(base, latest, "run_summary.json")
    if not os.path.exists(summary_path):
        print(f"\n  [WARNING] {phase_name.upper()} (run {latest}): No run_summary.json (may have crashed)")
        # Check if there are directories we can resolve
        sub_dir = os.path.join(base, latest)
        if os.path.isdir(sub_dir):
            partial = os.listdir(sub_dir)
            if partial:
                print(f"      Partial files: {', '.join(partial[:10])}")
        continue
    with open(summary_path) as f:
        try:
            summary = json.load(f)
            print(f"\n  [OK] {phase_name.upper()} (run {latest})")
            for k, v in summary.items():
                if isinstance(v, dict):
                    print(f"      {k}:")
                    for kk, vv in v.items():
                        print(f"        {kk}: {vv}")
                elif isinstance(v, list):
                    print(f"      {k}: {', '.join(str(x) for x in v)}")
                else:
                    print(f"      {k}: {v}")
        except Exception as e:
            print(f"\n  [ERROR] {phase_name.upper()} (run {latest}): Failed to parse run_summary.json: {e}")

print("\n" + "=" * 72)
print("  Pipeline execution finished!")
print("=" * 72)
print("\nRun Step 3 below to zip and download results.")

## Step 3: Zip and Download Results

In [ ]:
import zipfile
import datetime
import shutil
from google.colab import files

timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
zip_filename = f"colab_results_{timestamp}.zip"

print(f"Packaging results into {zip_filename}...")
with zipfile.ZipFile(zip_filename, "w", zipfile.ZIP_DEFLATED) as zipf:
    # Compress experiments results
    # Note: resolving the symlink if Google Drive was mounted
    if os.path.islink("experiments/results"):
        target_dir = os.readlink("experiments/results")
        for root, dirs, files_in_dir in os.walk(target_dir):
            for file in files_in_dir:
                full_path = os.path.join(root, file)
                rel_path = os.path.join("experiments/results", os.path.relpath(full_path, target_dir))
                zipf.write(full_path, rel_path)
    else:
        for root, dirs, files_in_dir in os.walk("experiments/results"):
            for file in files_in_dir:
                zipf.write(os.path.join(root, file))
    
    # Compress processed datasets
    for root, dirs, files_in_dir in os.walk("data/processed"):
        for file in files_in_dir:
            zipf.write(os.path.join(root, file))

print(f"Zip archive {zip_filename} created.")

# If Google Drive is mounted, copy zip to Drive as a persistent backup
if os.path.exists("/content/drive/MyDrive"):
    try:
        drive_backup = f"/content/drive/MyDrive/{zip_filename}"
        print(f"Saving backup copy to Google Drive: {drive_backup}...")
        shutil.copy(zip_filename, drive_backup)
        print("Backup saved to Google Drive successfully!")
    except Exception as e:
        print(f"Failed to backup to Google Drive: {e}")

print("Triggering browser download...")
try:
    files.download(zip_filename)
except Exception as e:
http://www.google.com
    print(f"Could not trigger browser download: {e}")
    print(f"Please download the file '{zip_filename}' directly from the files tab in Colab.")